<a href="https://colab.research.google.com/github/fuadsaniibrahim/FIP-Intern-Deliverable/blob/main/Module2_DL_TransferLearning_GradCAM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Module 2 (Part A): Advanced Deep Learning with Transfer Learning and Grad-CAM

**Techniques:** transfer learning (a ResNet-18 pretrained on ImageNet, fine-tuned in two stages), modern training practice (data augmentation, one-cycle learning-rate schedule, mixed-precision training), and model explanation with Grad-CAM.

**Task:** classify 10 everyday object classes (CIFAR-10: airplane, automobile, bird, cat, deer, dog, frog, horse, ship, truck).

**What this notebook does**
1. Loads CIFAR-10 and creates train / validation / test splits
2. Trains a small CNN **from scratch** as a baseline
3. Fine-tunes a **pretrained ResNet-18** in two stages (train the new head, then unfreeze everything)
4. Compares both on the held-out test set
5. Uses **Grad-CAM** to show which image regions drive a prediction
6. Wraps the model in a Gradio web app for the screen recording

**Use a GPU:** in Colab choose Runtime > Change runtime type > T4 GPU before running.

In [1]:
import os, time, copy, random, warnings
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
import torchvision
from torchvision import datasets, transforms, models
from PIL import Image
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

QUICK_RUN = False     # True = tiny dry run on fake data for debugging only. Leave False for real results.
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
use_amp = device == "cuda"
IMG = 64 if QUICK_RUN else 128          # images are upscaled from 32x32 so the pretrained network sees larger inputs
BATCH = 64 if QUICK_RUN else 128
EPOCHS_SCRATCH = 1 if QUICK_RUN else 6
EPOCHS_HEAD, EPOCHS_FINE = (1, 1) if QUICK_RUN else (1, 4)
CLASSES = ["airplane", "automobile", "bird", "cat", "deer", "dog", "frog", "horse", "ship", "truck"]
print("device:", device, "| torch", torch.__version__, "| quick run:", QUICK_RUN)

device: cpu | torch 2.11.0+cpu | quick run: False


## 1. Data
Training images get random crops and horizontal flips (augmentation). Validation and test images only get resizing and normalisation. The 50,000 training images are split into 45,000 for training and 5,000 for validation, so the **test set is touched only once, at the end**. Normalisation uses the ImageNet mean and standard deviation because the pretrained network expects it.

In [ ]:
MEAN, STD = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)
train_tf = transforms.Compose([transforms.RandomCrop(32, padding=4), transforms.RandomHorizontalFlip(),
                               transforms.Resize(IMG), transforms.ToTensor(), transforms.Normalize(MEAN, STD)])
eval_tf  = transforms.Compose([transforms.Resize(IMG), transforms.ToTensor(), transforms.Normalize(MEAN, STD)])

if QUICK_RUN:
    mk = lambda n, tf: datasets.FakeData(size=n, image_size=(3, 32, 32), num_classes=10, transform=tf, random_offset=0)
    full_train, full_val, test_set = mk(512, train_tf), mk(512, eval_tf), mk(256, eval_tf)
else:
    full_train = datasets.CIFAR10("data", train=True,  download=True, transform=train_tf)
    full_val   = datasets.CIFAR10("data", train=True,  download=True, transform=eval_tf)
    test_set   = datasets.CIFAR10("data", train=False, download=True, transform=eval_tf)

idx = np.random.RandomState(SEED).permutation(len(full_train))
n_val = 5000 if not QUICK_RUN else 128
train_set, val_set = Subset(full_train, idx[n_val:]), Subset(full_val, idx[:n_val])
loader = lambda ds, shuffle: DataLoader(ds, batch_size=BATCH, shuffle=shuffle, num_workers=2, pin_memory=use_amp)
train_dl, val_dl, test_dl = loader(train_set, True), loader(val_set, False), loader(test_set, False)
print(f"train {len(train_set)} | val {len(val_set)} | test {len(test_set)}")

 44%|████▍     | 74.7M/170M [21:42<32:33, 49.0kB/s]

In [ ]:
# a quick look at the augmented training images
x, y = next(iter(train_dl))
mean, std = torch.tensor(MEAN).view(3, 1, 1), torch.tensor(STD).view(3, 1, 1)
fig, ax = plt.subplots(2, 8, figsize=(14, 4))
for a, img, lab in zip(ax.ravel(), x[:16], y[:16]):
    a.imshow((img * std + mean).clamp(0, 1).permute(1, 2, 0)); a.set_title(CLASSES[int(lab)], fontsize=8); a.axis("off")
plt.tight_layout(); plt.show()

## 2. Training utilities
One training loop is reused for every model. Mixed precision (`autocast` and a gradient scaler) speeds up GPU training and is switched off automatically on CPU.

In [ ]:
scaler = torch.amp.GradScaler("cuda", enabled=use_amp)

def run_epoch(model, dl, opt=None, sched=None):
    train = opt is not None
    model.train(train)
    tot_loss = tot_correct = n = 0
    with torch.set_grad_enabled(train):
        for xb, yb in dl:
            xb, yb = xb.to(device, non_blocking=True), yb.to(device, non_blocking=True)
            with torch.autocast(device_type="cuda", enabled=use_amp):
                out = model(xb); loss = F.cross_entropy(out, yb)
            if train:
                opt.zero_grad(set_to_none=True)
                scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
                if sched is not None: sched.step()
            tot_loss += loss.item() * len(yb); tot_correct += (out.argmax(1) == yb).sum().item(); n += len(yb)
    return tot_loss / n, tot_correct / n

def fit(model, epochs, max_lr, name, history, params=None):
    params = [p for p in model.parameters() if p.requires_grad] if params is None else params
    opt = torch.optim.AdamW(params, lr=max_lr, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=max_lr, epochs=epochs, steps_per_epoch=len(train_dl))
    for ep in range(epochs):
        t = time.time()
        tl, ta = run_epoch(model, train_dl, opt, sched)
        vl, va = run_epoch(model, val_dl)
        history.append(dict(phase=name, epoch=len(history) + 1, train_loss=tl, train_acc=ta, val_loss=vl, val_acc=va))
        print(f"[{name}] epoch {ep+1}/{epochs}  train loss {tl:.3f} acc {ta:.3f} | val loss {vl:.3f} acc {va:.3f}  ({time.time()-t:.0f}s)")

## 3. Baseline: a small CNN trained from scratch

In [ ]:
def conv_block(i, o): return nn.Sequential(nn.Conv2d(i, o, 3, padding=1, bias=False), nn.BatchNorm2d(o), nn.ReLU(inplace=True), nn.MaxPool2d(2))

scratch = nn.Sequential(conv_block(3, 32), conv_block(32, 64), conv_block(64, 128), conv_block(128, 256),
                        nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Dropout(0.3), nn.Linear(256, 10)).to(device)
n_scratch = sum(p.numel() for p in scratch.parameters())
print(f"parameters: {n_scratch:,}")
hist_scratch = []
fit(scratch, EPOCHS_SCRATCH, 3e-3, "scratch", hist_scratch)

## 4. Transfer learning: pretrained ResNet-18, fine-tuned in two stages
**Stage 1 (head only):** the pretrained backbone is frozen and only the new 10-class output layer is trained. This stops random head weights from sending large, damaging gradients into good pretrained features.
**Stage 2 (full fine-tune):** all layers are unfrozen and trained with a one-cycle learning-rate schedule, which warms up and then anneals the learning rate.

In [ ]:
weights = None if QUICK_RUN else models.ResNet18_Weights.IMAGENET1K_V1
resnet = models.resnet18(weights=weights)
resnet.fc = nn.Linear(resnet.fc.in_features, 10)
resnet = resnet.to(device)
n_resnet = sum(p.numel() for p in resnet.parameters())
print(f"parameters: {n_resnet:,}")

hist_resnet = []
for p in resnet.parameters(): p.requires_grad = False          # stage 1: freeze the backbone
for p in resnet.fc.parameters(): p.requires_grad = True
fit(resnet, EPOCHS_HEAD, 1e-3, "stage 1: head only", hist_resnet)

for p in resnet.parameters(): p.requires_grad = True           # stage 2: unfreeze everything
fit(resnet, EPOCHS_FINE, 1e-3, "stage 2: full fine-tune", hist_resnet)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
for name, h in [("scratch CNN", hist_scratch), ("ResNet-18 transfer", hist_resnet)]:
    d = pd.DataFrame(h)
    ax[0].plot(range(1, len(d) + 1), d.val_acc, marker="o", label=name)
    ax[1].plot(range(1, len(d) + 1), d.train_acc, marker="o", label=f"{name} train")
    ax[1].plot(range(1, len(d) + 1), d.val_acc, marker="x", ls="--", label=f"{name} val")
ax[0].set_title("Validation accuracy per epoch"); ax[0].set_xlabel("epoch"); ax[0].legend()
ax[1].set_title("Train vs validation accuracy"); ax[1].set_xlabel("epoch"); ax[1].legend(fontsize=8)
plt.tight_layout(); plt.savefig("dl_training_curves.png", dpi=130); plt.show()

## 5. Final evaluation on the test set
The two models were compared on the validation set during training. The test set is used here once.

In [ ]:
@torch.no_grad()
def predict_all(model, dl):
    model.eval(); ys, ps = [], []
    for xb, yb in dl:
        with torch.autocast(device_type="cuda", enabled=use_amp):
            out = model(xb.to(device))
        ps.append(out.argmax(1).cpu()); ys.append(yb)
    return torch.cat(ys).numpy(), torch.cat(ps).numpy()

results = {}
for name, m in [("Small CNN from scratch", scratch), ("ResNet-18 transfer learning", resnet)]:
    y_true, y_pred = predict_all(m, test_dl)
    results[name] = dict(test_acc=(y_true == y_pred).mean(), y_true=y_true, y_pred=y_pred)

test_table = pd.DataFrame({k: {"test accuracy": round(v["test_acc"], 4)} for k, v in results.items()}).T
test_table["parameters"] = [n_scratch, n_resnet]
display(test_table); test_table.to_csv("dl_test_results.csv")

best = results["ResNet-18 transfer learning"]
print(classification_report(best["y_true"], best["y_pred"], target_names=CLASSES, digits=3, zero_division=0))
fig, ax = plt.subplots(figsize=(7, 6))
ConfusionMatrixDisplay(confusion_matrix(best["y_true"], best["y_pred"]), display_labels=CLASSES).plot(ax=ax, xticks_rotation=45, colorbar=False)
plt.title("ResNet-18 transfer learning: test confusion matrix"); plt.tight_layout(); plt.savefig("dl_confusion_matrix.png", dpi=130); plt.show()

## 6. Explainability with Grad-CAM
Grad-CAM shows **where** in the image the network looked. For the target class it takes the gradient of the class score with respect to the last convolutional feature maps, averages each map's gradient to get an importance weight, forms the weighted sum of the maps, and keeps only the positive part (ReLU). The result is a coarse heat map laid over the image. It checks whether the model is looking at the object or at the background.

In [ ]:
class GradCAM:
    def __init__(self, model, layer):
        self.model, self.acts, self.grads = model, None, None
        def fwd(m, i, o):
            self.acts = o
            o.register_hook(lambda g: setattr(self, "grads", g))
        layer.register_forward_hook(fwd)

    def __call__(self, x, class_idx=None):
        self.model.eval(); self.model.zero_grad(set_to_none=True)
        x = x.clone().requires_grad_(True)
        out = self.model(x)
        if class_idx is None: class_idx = int(out.argmax(1))
        out[0, class_idx].backward()
        w = self.grads.mean(dim=(2, 3), keepdim=True)
        cam = F.relu((w * self.acts).sum(1)).squeeze(0).detach().cpu().numpy()
        cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
        return cam, class_idx, out.softmax(1).detach().cpu().squeeze(0)

resnet.float().eval()
cam_tool = GradCAM(resnet, resnet.layer4)

def overlay(img_pil, cam, alpha=0.45):
    base = np.array(img_pil.resize((256, 256))).astype(np.float32) / 255
    heat = np.array(Image.fromarray((cam * 255).astype(np.uint8)).resize((256, 256), Image.BILINEAR)) / 255
    colour = plt.get_cmap("jet")(heat)[..., :3]
    return Image.fromarray((((1 - alpha) * base + alpha * colour) * 255).astype(np.uint8))

def tensor_to_pil(t): return Image.fromarray(((t.cpu() * std + mean).clamp(0, 1).permute(1, 2, 0).numpy() * 255).astype(np.uint8))

fig, ax = plt.subplots(2, 6, figsize=(14, 5))
for k in range(6):
    xb, yb = test_set[k * 7]
    cam, pred, prob = cam_tool(xb.unsqueeze(0).to(device))
    img = tensor_to_pil(xb)
    ax[0, k].imshow(img.resize((256, 256))); ax[0, k].set_title(f"true: {CLASSES[yb]}", fontsize=9); ax[0, k].axis("off")
    ax[1, k].imshow(overlay(img, cam)); ax[1, k].set_title(f"pred: {CLASSES[pred]} ({prob[pred]:.2f})", fontsize=9); ax[1, k].axis("off")
plt.tight_layout(); plt.savefig("dl_gradcam_examples.png", dpi=130); plt.show()

In [ ]:
torch.save(resnet.state_dict(), "resnet18_cifar10.pt")
print("saved resnet18_cifar10.pt")

## 7. Working application (Gradio)
Upload any photo of one of the 10 classes and the app shows the top predictions plus the Grad-CAM heat map. Run the cell and use the public link for the screen recording. Photos that are not one of these classes will still get a prediction, because a classifier can only choose among the classes it knows.

In [ ]:
import gradio as gr
app_tf = transforms.Compose([transforms.Resize((IMG, IMG)), transforms.ToTensor(), transforms.Normalize(MEAN, STD)])

def classify(img):
    if img is None: return {}, None
    img = img.convert("RGB")
    x = app_tf(img).unsqueeze(0).to(device)
    cam, pred, prob = cam_tool(x)
    top = {CLASSES[i]: float(prob[i]) for i in prob.argsort(descending=True)[:3]}
    return top, overlay(img, cam)

demo = gr.Interface(
    fn=classify,
    inputs=gr.Image(type="pil", label="Upload an image"),
    outputs=[gr.Label(num_top_classes=3, label="Top predictions"), gr.Image(label="Grad-CAM: where the model looked")],
    title="CIFAR-10 classifier: ResNet-18 transfer learning with Grad-CAM",
    description="Classes: " + ", ".join(CLASSES) + ".",
    flagging_mode="never")

if QUICK_RUN:
    sample, _ = test_set[0]
    print(classify(tensor_to_pil(sample))[0])      # dry run: call the function directly
else:
    demo.launch(share=True)

## 8. Findings

In [ ]:
sc, rn = results["Small CNN from scratch"]["test_acc"], results["ResNet-18 transfer learning"]["test_acc"]
d = pd.DataFrame(hist_resnet)
print(f"Small CNN from scratch ({n_scratch:,} parameters, {EPOCHS_SCRATCH} epochs): test accuracy {sc:.1%}")
print(f"ResNet-18 transfer learning ({n_resnet:,} parameters, {EPOCHS_HEAD} head epoch + {EPOCHS_FINE} fine-tune epochs): test accuracy {rn:.1%}")
print(f"Difference: {100*(rn-sc):+.1f} percentage points on {len(test_set):,} test images")
print(f"After stage 1 (head only) validation accuracy was {d.val_acc.iloc[EPOCHS_HEAD-1]:.1%}; after full fine-tuning it was {d.val_acc.iloc[-1]:.1%}.")
wrong = confusion_matrix(best['y_true'], best['y_pred']); np.fill_diagonal(wrong, 0)
i, j = np.unravel_index(wrong.argmax(), wrong.shape)
print(f"Most common confusion: true {CLASSES[i]} predicted as {CLASSES[j]} ({wrong[i, j]} images).")

### Discussion
* **Transfer learning vs training from scratch.** The printed comparison above is the main result. A pretrained network starts with general visual features (edges, textures, shapes) learned from over a million ImageNet images, so it needs far fewer epochs to do well on a new task than a network starting from random weights.
* **Why two stages.** Training the new head first on frozen features avoids large random gradients disturbing the pretrained layers. The jump in validation accuracy between stage 1 and stage 2 shows how much adapting the backbone added.
* **Grad-CAM as a sanity check.** The heat maps show whether the network focuses on the object or on background cues. Spurious background reliance would be a warning sign even when accuracy is high. The maps are coarse because the last convolutional layer has low spatial resolution.
* **Most common confusion** (printed above) is usually between visually similar classes such as cats and dogs, which matches what a human would find hard at 32 by 32 pixels.

### Limitations
* CIFAR-10 images are only 32 by 32 pixels and are upscaled to feed the pretrained network, so fine detail is limited.
* Results come from a single training run with one random seed. Differences of a fraction of a percentage point would not be meaningful.
* Hyperparameters (learning rate, epochs) were set by hand and not tuned.
* The app classifies only the 10 training classes and gives no "none of these" answer.